In [2]:
!pip install \
    langchain \
    langchain-huggingface \
    langchain-community \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers

INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0rc2
    Uninstalling packaging-26.0rc2:
      Successfully uninstalled packaging-26.0rc2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoModel, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline

print("Transformers + LangChain ready ✅")
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Transformers + LangChain ready ✅
Torch: 2.8.0+cu126
CUDA: True


In [4]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

In [5]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load Model and Related Parameters
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

## Reusuable PromptTemplate Creation

In [6]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.5,
    top_p=0.2,
    top_k=1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)


Device set to use cuda:0
/tmp/ipykernel_55/1524294560.py:15: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [7]:
template = """Tell me a {adjective} joke about {content}.
"""
prompt = PromptTemplate.from_template(template)
prompt 

PromptTemplate(input_variables=['adjective', 'content'], input_types={}, partial_variables={}, template='Tell me a {adjective} joke about {content}.\n')

In [8]:
prompt.format(adjective="funny", content="chickens")

'Tell me a funny joke about chickens.\n'

# ------------------------------------------------------------------

In [9]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Define a function to ensure proper formatting
def format_prompt(variables):
    return prompt.format(**variables)

In [10]:
template = """Tell me a {adjective} story about {content}."""

prompt = PromptTemplate.from_template(template)
prompt 

# Create the chain with explicit formatting
joke_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
response = joke_chain.invoke({"adjective": "sad", "content": "boy"})
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.




Once upon a time, in a small village nestled between the rolling hills and the deep, dark forest, there lived a young boy named Elias. He was a kind and gentle soul, always helping others and spreading joy wherever he went. But despite his cheerful disposition, Elias harbored a deep sadness within him.

You see, Elias was an orphan. He had been found as a baby, abandoned and alone, in the forest near the village. The kind-hearted villagers had taken him in and raised him as their own. But no matter how much love and care they gave him, Elias couldn't shake the feeling that he didn't truly belong. He longed for the comfort and security of a family, a place where he could call his own.

As Elias grew older, he became a skilled craftsman, known for his beautiful wooden toys and intricate carvings. He would spend hours in his workshop, lost in thought as he worked, the sadness in his heart fueling his creativity. But no matter how many toys he made or how many villagers he brought joy wi

## Text Summarization

In [14]:
content = """
    The rapid advancement of technology in the 21st century has transformed various industries, including healthcare, education, and transportation. 
    Innovations such as artificial intelligence, machine learning, and the Internet of Things have revolutionized how we approach everyday tasks and complex problems. 
    For instance, AI-powered diagnostic tools are improving the accuracy and speed of medical diagnoses, while smart transportation systems are making cities more efficient and reducing traffic congestion. 
    Moreover, online learning platforms are making education more accessible to people around the world, breaking down geographical and financial barriers. 
    These technological developments are not only enhancing productivity but also contributing to a more interconnected and informed society.
"""

template = """Summarize the {content} in one sentence.
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
summarize_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
summary = summarize_chain.invoke({"content": content})
print(summary)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


The 21st century's technological innovations, including AI, machine learning, and the Internet of Things, are revolutionizing healthcare, education, and transportation by improving accuracy, efficiency, accessibility, and interconnectivity.


### Question answering

In [ ]:
content = """
    The solar system consists of the Sun, eight planets, their moons, dwarf planets, and smaller objects like asteroids and comets. 
    The inner planets—Mercury, Venus, Earth, and Mars—are rocky and solid. 
    The outer planets—Jupiter, Saturn, Uranus, and Neptune—are much larger and gaseous.
"""

question = "Which planets in the solar system are rocky and solid?"

template = """
    Answer the {question} based on the {content}.
    Respond "Unsure about answer" if not sure about the answer.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
qa_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
answer = qa_chain.invoke({"question": question, "content": content})
print(answer)

## Text Classifiation

In [11]:
text = """
    The concert last night was an exhilarating experience with outstanding performances by all artists.
"""

categories = "Entertainment, Food and Dining, Technology, Literature, Music."

template = """
    Classify the {text} into one of the {categories}.
    
    Category:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
classification_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Run the chain
category = classification_chain.invoke({"text": text, "categories": categories})
print(category)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    The concert last night was an exhilarating experience with outstanding performances by all artists.

    Category: Music.


## Code Generation

In [12]:
description = """
    Retrieve the names and email addresses of all customers from the 'customers' table who have made a purchase in the last 30 days. 
    The table 'purchases' contains a column 'purchase_date'
"""

template = """
    Generate an SQL query based on the {description}
    
    SQL Query:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
sql_generation_chain = (
    RunnableLambda(format_prompt) 
    | llm 
    | StrOutputParser()
)

# Run the chain
sql_query = sql_generation_chain.invoke({"description": description})
print(sql_query)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


    SELECT c.name, c.email
    FROM customers AS c
    INNER JOIN purchases AS p ON c.customer_id = p.customer_id
    WHERE p.purchase_date >= DATEADD(day, -30, GETDATE())


## Role Playing

- `role`: Specifies the character, expertise, or persona the LLM should embody
- `tone`: Defines the communication style and emotional quality of responses
- `question`: Contains the user's query that needs addressing

In [13]:
role = """
    Counter Strike game master
"""

tone = "engaging and immersive"

template = """
    You are an expert {role}. I have this question {question}. I would like our conversation to be {tone}.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
roleplay_chain = (
    RunnableLambda(format_prompt)
    | llm 
    | StrOutputParser()
)

# Create an interactive chat loop
while True:
    query = input("Question: ")
    
    if query.lower() in ["quit", "exit", "bye"]:
        print("Answer: Goodbye!")
        break
        
    response = roleplay_chain.invoke({"role": role, "question": query, "tone": tone})
    print("Answer: ", response)

Question:  What is the best powerful gun in that game?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Answer:      Greetings, esteemed inquirer! I'm thrilled to hear that you're interested in the world of Counter-Strike and its vast array of weaponry. It's a question I get asked frequently, and I'm always happy to share my insights with fellow enthusiasts.

    When it comes to the most powerful guns in Counter-Strike, there are a few contenders that consistently rise to the top. Let's take a look at some of the heavy hitters and what makes them so formidable.

    1. AWP (Awp-25): The AWP, or the Arctic Warfare Magnum, is arguably the most iconic and powerful weapon in Counter-Strike. It's a sniper rifle that deals massive damage with a single shot to the head or chest, making it an invaluable asset for any team. Its high accuracy and long range make it a formidable weapon for taking out enemies from a distance. However, its high cost and limited ammo capacity make it a risky investment.

    2. M249: The M249 is a light machine gun that is known


Question:  The lightest gun ?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Answer:      Greetings, esteemed inquirer! I'm thrilled to help answer your question about the lightest gun in Counter-Strike. This game, with its rich history and intricate details, never fails to surprise and delight us, doesn't it?

    Now, let's dive into the world of Counter-Strike and uncover the answer to your query. The lightest gun in Counter-Strike: Global Offensive (CS:GO) is the USP-S, or the Ultra-Silenced Pistol. This pistol, a favorite among many players for its accuracy and silenced firing capability, weighs in at a mere 1.65 pounds (or 0.75 kilograms).

    But let's not forget that the weight of a gun isn't the only factor to consider when choosing a weapon in CS:GO. Other aspects, such as damage output, rate of fire, accuracy, and cost, all play crucial roles in determining the overall effectiveness of a weapon.

    So, while the USP-S may be the lightest gun in CS:GO, it's essential to consider how it fits into


Question:  bye


Answer: Goodbye!


## Exercise

In [14]:

# Create the prompt template
template = """
Analyze the following product review:
"{review}"

Provide your analysis in the following format:
- Sentiment: (positive, negative, or neutral)
- Key Features Mentioned: (list the product features mentioned)
- Summary: (one-sentence summary)
"""

product_review_prompt = PromptTemplate.from_template(template)

# Create a formatting function
def format_review_prompt(variables):
    return product_review_prompt.format(**variables)

# Build the LCEL chain
review_analysis_chain = (
    RunnableLambda(format_review_prompt)
    | llm 
    | StrOutputParser()
)

# Process the reviews
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

for i, review in enumerate(reviews):
    print(f"==== Review #{i+1} ====")
    result = review_analysis_chain.invoke({"review": review})
    print(result)
    print()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


==== Review #1 ====


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Analysis:
- Sentiment: Positive
- Key Features Mentioned: Camera quality, battery life, heating up during gaming
- Summary: The reviewer expresses their love for the smartphone, praising its exceptional camera quality and all-day battery life. However, they note a downside of the device heating up during gaming.

==== Review #2 ====

Analysis:
- Sentiment: Negative
- Key Features Mentioned: Speed, crashes, keyboard
- Summary: The reviewer expresses dissatisfaction with the laptop's slow performance, frequent crashes, and a malfunctioning keyboard, as well as their negative experience with customer service.

